In [142]:
from pyspark.sql import SparkSession

spark=SparkSession.builder \
     .appName("working with multiple files") \
     .getOrCreate()

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
patients_df=spark.read.option("header",True).csv("patients.csv",inferSchema=True)
patients_df.show()

In [ ]:
appointments_df=spark.read.option("header",True).csv("appointments.csv",inferSchema= True)
appointments_df.show()

In [ ]:
preferences_df=spark.read.option("multiline",True).json("patient_preferences.json")
preferences_df.show()

PART 1 - CSV **Ingestion**

In [ ]:
patients_df.printSchema()

In [ ]:
appointments_df.printSchema()

In [ ]:
patients_df.count()

In [ ]:
appointemnts_df.count()

In [ ]:
patients_df.show(5)

In [ ]:
patients_df.select("city").distinct().show()

In [ ]:
appointments_df.select("department").distinct().show()

In [ ]:
from pyspark.sql.functions import *

patients_df.write.mode("overwrite").parquet("patients_parquet")


In [ ]:
parquet_df=spark.read.parquet("patients_parquet")

In [ ]:
patients_df.count()
parquet_df.count()

PART 2-**FILTERING**

In [ ]:
patients_df.filter(col("city") == "Hyderabad").show()

In [ ]:
patients_df.filter(col("gender") == "Female").show()

In [ ]:
patients_df.filter(col("age")>40).show()

In [ ]:
patients_df.filter(col("insurance_status")=="Active").show()

In [ ]:
patients_df.filter(col("insurance_status")=="Inactive").show()

In [ ]:
patients_df.filter(col("blood_group")=="O+").show()

In [ ]:
appointments_df.filter(col("department")=="Cardiology").show()

PART 3 - Null **Handling**

In [ ]:
patients_df.filter(col("city").isNull()).show()

In [ ]:
patients_df.filter(col("blood_group").isNull()).show()

In [ ]:
appointments_df.filter(col("consultation_fee").isNull()).show()

In [ ]:
from pyspark.sql.functions import count,when

patients_df.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in patients_df.columns
])

In [ ]:
patients_df = patients_df.fillna({"city":"Unknown"})


In [ ]:
patients_df = patients_df.fillna({"blood_group":"Not Available"})


In [ ]:
appointments_df = appointments_df.fillna({"consultation_fee":0})


In [ ]:
appointments_df.na.drop(subset=["consultation_fee"]).show()


In [ ]:
patients_df = patients_df.withColumn(
    "data_quality_status",
    when(
        col("city").isNull() |
        col("blood_group").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)
patients_df.groupBy("data_quality_status").count().show()

Part 4 - Built-In **Functions**

In [ ]:
patients_df.select(upper("patient_name")).show()

In [ ]:
patients_df.select(lower("patient_name")).show()

In [ ]:
patients_df.select("patient_name",length("patient_name").alias("length")).show()

In [ ]:
patients_df.select(
    substring("patient_name",1,3).alias("first3")
).show()

In [ ]:
patients_df=patients_df.withColumn(
    "age_group",
    when(col("age")<30,"Young")
    .when(col("age")<50,"Adult")
    .otherwise("Senior")
)

In [ ]:
patients_df=patients_df.withColumn(
    "insurance_flag",
    when(col("insurance_status")=="Active",1).otherwise(0)
)

In [ ]:
patients_df=patients_df.withColumn(
    "senior_citizen",
    when(col("age")>=60,"Yes").otherwise("No")
)

In [ ]:
patients_df.select(
    concat_ws("-","patient_name","city")
).show()

In [ ]:
patients_df=patients_df.withColumn(
    "patient_name",
    trim(col("patient_name"))
)

In [ ]:
patients_df=patients_df.withColumn(
    "city_upper",
    upper(col("city"))
)

PART 5-GroupBy & Aggregrations

In [ ]:
patients_df.groupBy("city").count().show()

In [ ]:
patients_df.groupBy("gender").count().show()

In [ ]:
patients_df.groupBy("blood_group").count().show()

In [ ]:
appointments_df.groupBy("department").count().show()

In [ ]:
patients_df.groupBy("city").avg("age").show()

In [ ]:
patients_df.groupBy("city").max("age").show()

In [ ]:
patients_df.groupBy("city").min("age").show()


In [ ]:
appointments_df.groupBy("department") \
    .avg("consultation_fee").show()


In [ ]:
appointments_df.groupBy("department") \
    .sum("consultation_fee").show()


In [ ]:
appointments_df.groupBy("department") \
    .sum("consultation_fee") \
    .orderBy(desc("sum(consultation_fee)")) \
    .show(1)

Part 6 - Joins

In [ ]:
inner_df=patients_df.join(
    appointments_df,
    "patient_id",
    "inner"
).show()

In [ ]:
left_df=patients_df.join(
    appointments_df,
    "patient_id",
    "left"
)
left_df.show()

In [ ]:
right_df=patients_df.join(
    appointments_df,
    "patient_id",
    "right"
)
right_df.show()

In [ ]:
full_df=patients_df.join(
    appointments_df,
    "patient_id",
    "full"
).show()

In [ ]:
left_df.filter(
    appointments_df.patient_id.isNull()
).show()

In [ ]:
right_df.filter(
    appointments_df.patient_id.isNull()
).show()


In [ ]:
appointments_df.groupBy("patient_id") \
.count().show()

In [ ]:
appointments_df.groupBy("patient_id") \
    .sum("consultation_fee").show()

In [ ]:
appointments_df.groupBy("patient_id") \
    .sum("consultation_fee") \
    .orderBy(desc("sum(consultation_fee)")) \
    .show(1)


In [ ]:
appointments_df.groupBy("patient_id") \
    .count() \
    .show()

PART 7-Window Functions

In [ ]:
from pyspark.sql.window import Window

spending_df=appointments_df.groupBy("patient_id")\
    .sum("consultation_fee") \
    .withColumnRenamed(
        "sum(consultation_fee)",
        "total_fees"
    )
window_spec=Window.orderBy(desc("total_fees"))

spending_df=spending_df.withColumn(
    "rank",
    rank().over(window_spec)
)
spending_df.show()


In [ ]:
spending_df = spending_df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
)


In [ ]:
spending_df = spending_df.withColumn(
    "row_number",
    row_number().over(window_spec)
)


In [ ]:
spending_df.show()


In [ ]:
spending_df.orderBy(desc("total_fees")).show(1)

In [ ]:
spending_df.orderBy(desc("total_fees")).show(3)

In [ ]:
joined = spending_df.join(
    patients_df,
    "patient_id"
)

In [ ]:
city_window = Window.partitionBy("city") \
    .orderBy(desc("total_fees"))

In [ ]:
joined.withColumn(
    "rank",
    rank().over(city_window)
).filter(col("rank")==1).show()

In [ ]:
running_window=Window.orderBy("patient_id")

In [ ]:
spending_df.withColumn(
    "running_total",
    sum("total_fees").over(running_window)
).show()

In [ ]:
spending_df.withColumn(
    "lead_fee",
    lead("total_fees").over(window_spec)
).show()


In [ ]:
spending_df.withColumn(
    "lag_fee",
    lag("total_fees").over(window_spec)
).show()

PART 8-JSON Processing

In [ ]:
preferences_df.printSchema()

In [ ]:
preferences_df.select(
    "patient_id",
    col("contact.phone").alias("phone")
).show()

In [ ]:
preferences_df.select(
    "patient_id",
    col("contact.email").alias("email")
).show()

In [ ]:
preferences_df.filter(
    col("contact.phone").isNull()
).show()

In [ ]:
preferences_df.filter(
    col("contact.email").isNull()
).show()

In [ ]:
preferences_df.filter(
    col("preferred_hospital").isNull()
).show()


In [ ]:
preferences_df = preferences_df.fillna(
    {"preferred_hospital":"Unknown"}
)


In [ ]:
preferences_df = preferences_df.withColumn(
    "phone",
    when(col("contact.phone").isNull(),
         "Not Available")
    .otherwise(col("contact.phone"))
)

In [ ]:
preferences_df = preferences_df.withColumn(
    "email",
    when(col("contact.email").isNull(),
         "Not Available")
    .otherwise(col("contact.email"))
)

In [ ]:
patients_pref_df = patients_df.join(
    preferences_df,
    "patient_id",
    "left"
)

patients_pref_df.show()

In [ ]:
patients_df.createOrReplaceTempView("patients")

In [ ]:
appointments_df.createOrReplaceTempView("appointments")

In [ ]:
spark.sql("""
SELECT * FROM patients
""").show()

In [ ]:
spark.sql("""
SELECT *
FROM patients
WHERE city='Hyderabad'
""").show()

In [ ]:
spark.sql("""
SELECT city,COUNT(*)
FROM patients
GROUP BY city
""").show()

In [ ]:
spark.sql("""
SELECT department,COUNT(*)
FROM appointments
GROUP BY department
""").show()

In [ ]:
spark.sql("""
SELECT department,
AVG(consultation_fee)
FROM appointments
GROUP BY department
""").show()

In [ ]:
spark.sql("""
SELECT MAX(consultation_fee)
FROM appointments
""").show()

In [ ]:
spark.sql("""
SELECT patient_id,
COUNT(*) appointment_count
FROM appointments
GROUP BY patient_id
""").show()


In [ ]:
spark.sql("""
SELECT patient_id,
SUM(consultation_fee) total_spent
FROM appointments
GROUP BY patient_id
ORDER BY total_spent DESC
LIMIT 5
""").show()

PART-10-ETL PROJECT

In [ ]:
patients_df = patients_df.fillna({
    "city":"Unknown",
    "blood_group":"Not Available"
})

In [ ]:
appointments_df = appointments_df.fillna({
    "consultation_fee":0
})

In [ ]:
final_df = patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).join(
    preferences_df,
    "patient_id",
    "left"
)

In [ ]:
final_df = final_df.withColumn(
    "age_group",
    when(col("age")<30,"Young")
    .when(col("age")<50,"Adult")
    .otherwise("Senior")
)

In [ ]:
patient_spending = final_df.groupBy(
    "patient_id",
    "patient_name"
).agg(
    sum("consultation_fee")
    .alias("total_spending")
)


In [ ]:
department_revenue = final_df.groupBy(
    "department"
).agg(
    sum("consultation_fee")
    .alias("department_revenue")
)

In [ ]:
final_df.write.mode("overwrite") \
    .parquet("hospital_analytics_output")

print("Hospital Analytics ETL Completed Successfully")

In [ ]:
print("Top Spending Patients")
patient_spending.orderBy(
    desc("total_spending")
).show()



In [ ]:
print("Department Revenue")
department_revenue.orderBy(
    desc("department_revenue")
).show()

In [ ]:
print("Total Patients")
print(patients_df.count())

In [ ]:
print("Total Appointments")
print(appointments_df.count())

print("Completed Appointments")
appointments_df.filter(
    col("status")=="Completed"
).count()